## Check Enriched Table

In [0]:
%sql
SELECT *
FROM olist.silver.orders_enriched
LIMIT 20

## KPI 1 — Monthly Revenue Trend

In [0]:
%sql
-- How is revenue trending over time?
-- Tracks revenue, order volume, true AOV, satisfaction, and unique customers per month.
-- Customer count uses customer_unique_id (the stable person identifier),
-- not customer_id (which is per-order in Olist).
CREATE OR REPLACE TABLE olist.gold.monthly_revenue AS
SELECT
    oe.order_purchase_year,
    oe.order_purchase_month,
    ROUND(SUM(oe.total_item_value), 2)        AS monthly_revenue,
    COUNT(DISTINCT oe.order_id)               AS monthly_orders,
    ROUND(SUM(oe.total_item_value) / NULLIF(COUNT(DISTINCT oe.order_id), 0), 2) AS monthly_avg_order_value,
    ROUND(AVG(oe.avg_review_score), 1)        AS monthly_avg_review_score,
    COUNT(DISTINCT c.customer_unique_id)      AS monthly_active_customers
FROM olist.silver.orders_enriched oe
JOIN olist.silver.customers c
    ON oe.customer_id = c.customer_id
GROUP BY oe.order_purchase_year, oe.order_purchase_month
ORDER BY oe.order_purchase_year, oe.order_purchase_month

In [0]:
%sql
SELECT *
FROM olist.gold.monthly_revenue

## KPI 2 — Seller Performance

In [0]:
%sql
-- Which sellers are best/worst?
-- Ranks sellers by revenue, satisfaction, delivery speed, and late delivery rate.
-- unique_customers_served uses customer_unique_id (real people), not customer_id.
-- late_delivery_pct = late / delivered (excludes pending/canceled from denominator).
CREATE OR REPLACE TABLE olist.gold.seller_performance AS
SELECT
    oe.seller_id,
    oe.seller_city,
    oe.seller_state,
    COUNT(DISTINCT oe.order_id)               AS total_orders,
    COUNT(DISTINCT CASE WHEN oe.order_delivered_customer_date IS NOT NULL
                        THEN oe.order_id END) AS delivered_orders,
    ROUND(SUM(oe.total_item_value), 2)        AS total_revenue,
    ROUND(SUM(oe.total_item_value) / NULLIF(COUNT(DISTINCT oe.order_id), 0), 2) AS avg_order_value,
    ROUND(AVG(oe.avg_review_score), 1)        AS avg_review_score,
    COUNT(DISTINCT c.customer_unique_id)      AS unique_customers_served,
    ROUND(AVG(oe.delivery_time_days), 1)      AS avg_delivery_days,
    COUNT(DISTINCT CASE WHEN oe.is_delivered_late THEN oe.order_id END) AS late_orders,
    ROUND(
        COUNT(DISTINCT CASE WHEN oe.is_delivered_late THEN oe.order_id END) * 100.0
        / NULLIF(COUNT(DISTINCT CASE WHEN oe.order_delivered_customer_date IS NOT NULL
                                     THEN oe.order_id END), 0),
        1
    ) AS late_delivery_pct
FROM olist.silver.orders_enriched oe
JOIN olist.silver.customers c
    ON oe.customer_id = c.customer_id
GROUP BY oe.seller_id, oe.seller_city, oe.seller_state
ORDER BY total_revenue DESC

In [0]:
%sql
SELECT *
FROM olist.gold.seller_performance
LIMIT 10

## KPI 3 — Product Category Performance

In [0]:
%sql
-- Which product categories drive the most revenue?
-- Tracks orders, units, revenue, satisfaction, and shipping cost burden by category and month.
-- COALESCE turns NULL categories into 'Unknown' so they render clearly in BI tools.
CREATE OR REPLACE TABLE olist.gold.product_category_analysis AS
SELECT
    COALESCE(product_category_name_english, 'Unknown') AS product_category_name_english,
    order_purchase_year,
    order_purchase_month,
    COUNT(DISTINCT order_id)               AS total_orders,
    COUNT(*)                               AS total_units_sold,
    ROUND(SUM(total_item_value), 2)        AS total_revenue,
    ROUND(AVG(total_item_value), 2)        AS avg_item_value,
    ROUND(AVG(avg_review_score), 1)        AS avg_review_score,
    ROUND(AVG(freight_ratio), 3)           AS avg_freight_ratio
FROM olist.silver.orders_enriched
GROUP BY
    COALESCE(product_category_name_english, 'Unknown'),
    order_purchase_year,
    order_purchase_month
ORDER BY total_revenue DESC

In [0]:
%sql
SELECT *
FROM olist.gold.product_category_analysis
LIMIT 10
    


## KPI 4 — Delivery Performance by Region + Preview

In [0]:
%sql
-- Where are deliveries slow?
-- Shows delivery time, promise gap, and late rate by customer state and month.
-- avg_estimated_vs_actual: negative = delivering early, positive = delivering late
-- late_delivery_pct = late / delivered (excludes pending/canceled from denominator).
CREATE OR REPLACE TABLE olist.gold.delivery_performance AS
SELECT
    customer_state,
    order_purchase_year,
    order_purchase_month,
    COUNT(DISTINCT order_id)               AS total_orders,
    COUNT(DISTINCT CASE WHEN order_delivered_customer_date IS NOT NULL
                        THEN order_id END) AS delivered_orders,
    ROUND(AVG(delivery_time_days), 1)      AS avg_delivery_days,
    ROUND(AVG(estimated_vs_actual_days), 1) AS avg_estimated_vs_actual,
    COUNT(DISTINCT CASE WHEN is_delivered_late THEN order_id END) AS late_orders,
    ROUND(
        COUNT(DISTINCT CASE WHEN is_delivered_late THEN order_id END) * 100.0
        / NULLIF(COUNT(DISTINCT CASE WHEN order_delivered_customer_date IS NOT NULL
                                     THEN order_id END), 0),
        1
    ) AS late_delivery_pct
FROM olist.silver.orders_enriched
GROUP BY customer_state, order_purchase_year, order_purchase_month
ORDER BY late_delivery_pct DESC

In [0]:
%sql
SELECT *
FROM olist.gold.delivery_performance
LIMIT 10

## KPI 5 — Customer Segmentation + Preview

In [0]:
%sql
-- Customer-level metrics aggregated by the stable unique person ID.
-- One row per customer. State carried forward as the state of their most recent order.
CREATE OR REPLACE TABLE olist.gold.customer_segments AS
WITH customer_metrics AS (
    SELECT
        c.customer_unique_id,
        MAX_BY(oe.customer_state, oe.order_purchase_date) AS customer_state,
        COUNT(DISTINCT oe.order_id)            AS total_orders,
        ROUND(SUM(oe.total_item_value), 2)     AS total_spent,
        ROUND(SUM(oe.total_item_value) / NULLIF(COUNT(DISTINCT oe.order_id), 0), 2) AS avg_order_value,
        ROUND(AVG(oe.avg_review_score), 1)     AS avg_review_score,
        MIN(oe.order_purchase_date)            AS first_order_date,
        MAX(oe.order_purchase_date)            AS last_order_date
    FROM olist.silver.orders_enriched oe
    JOIN olist.silver.customers c
        ON oe.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
)
SELECT
    *,
    CASE
        WHEN total_orders = 1 THEN 'One-time'
        WHEN total_orders BETWEEN 2 AND 3 THEN 'Repeat'
        WHEN total_orders >= 4 THEN 'VIP'
    END AS customer_segment
FROM customer_metrics

In [0]:
%sql
-- Sanity check: counts per segment
SELECT
    customer_segment,
    COUNT(*) AS customers_in_segment
FROM olist.gold.customer_segments
GROUP BY customer_segment
ORDER BY customers_in_segment DESC;

In [0]:
%sql
SELECT *
FROM olist.gold.customer_segments
LIMIT 10

## Validate All Gold Tables

In [0]:
# Validate all Gold tables
gold_tables = [
    "monthly_revenue", "seller_performance", 
    "product_category_analysis", "delivery_performance",
    "customer_segments"
]

for t in gold_tables:
    count = spark.table(f"olist.gold.{t}").count()
    print(f"olist.gold.{t}: {count} rows")